<a href="https://colab.research.google.com/github/FabioContrera/criando-agentes-de-ia-com-agno/blob/main/Aula%205/Aula_5_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 5.1 — Do protótipo à produção: por que AgentOS?

**Curso:** Agno: Criando agentes e sistemas multiagente

**Aula:** 5 — Gerenciando agentes com AgentOS

**Instrutor:** Fabio Contrera

---

## Onde estamos

Ao final da **Aula 4**, o ScoutAI FC ficou "**completo**":
- time conversacional,
- workflow estruturado.

Mas, faltam coisas que **só o ambiente de produção entrega**:

- **Persistência**
- **Observabilidade**
- **Servir como API**
- **Gestão centralizada**
- **Isolamento por usuário**

Plano:

1. **Sentir a dor** do "modo notebook" tentando ir pra produção
2. Conceito de **AgentOS** como sistema operacional pra agentes
3. Configurar o **mínimo viável** com `SqliteDb` + `AgentOS` + FastAPI
4. Plugar um agente do ScoutAI FC ao AgentOS
5. Subir a **UI do AgentOS ao vivo**.



## Setup

Adicionamos `agno[os]` (que traz tudo necessário pro AgentOS, incluindo FastAPI e dependências de banco).


In [ ]:
!pip install -U "agno[os]" openai tavily-python wikipedia pydantic

In [ ]:
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")

---

## Passo 1 — A dor: "modo notebook" não vira produto

O que **falta** ao ScoutAI FC pra ele virar produto:

| Capacidade necessária em produção | Status no notebook |
|---|---|
| Conversa do usuário guardada entre sessões | ❌ Some quando o kernel reinicia |
| Quem fez qual pergunta, quando | ❌ Sem registro estruturado |
| Custo total e custo por turno | ❌ Visível só no log do `print_response` |
| Tempo de execução de cada agente | ❌ Não rastreado |
| Sessões isoladas por usuário | ⚠️ Funciona via `session_id`, mas em memória |
| Memória que persiste entre reinícios | ❌ Tudo perdido ao reiniciar |



---

## Passo 2 — O que é o AgentOS?

Pensa no AgentOS como um **sistema operacional pra agentes**.

| AgentOS |
|---|
| Gerencia execuções de agentes/times/workflows |
| Guarda sessões, conversas, memórias persistentes |
| Tools, knowledge bases, MCP servers |
| Tracing de agentes, métricas, custos |
| API REST + dashboard visual |
| Isolamento por `user_id` e `session_id` |

<br>

> **Ponto importante:** o AgentOS roda **na sua infra**, não no servidor da Agno.


---

## Passo 3 — Adicionando banco de dados a um agente


Quando você passa `db=...` a um agente, ele automaticamente:

- **Guarda cada execução**
- **Mantém sessões**
- **Habilita memórias persistentes**


In [ ]:
from agno.agent import Agent
from agno.models.openai import OpenAIChat
from agno.db.sqlite import SqliteDb
from agno.tools.tavily import TavilyTools
from agno.tools.wikipedia import WikipediaTools

# Banco SQLite — arquivo local, sem servidor
db = SqliteDb(db_file="/tmp/scoutai.db")

modelo_treinador = OpenAIChat(id="gpt-5.4")

# Agente Treinador, agora com banco
treinador = Agent(
    name="Treinador",
    role="Voz oficial do ScoutAI FC para o usuário",
    model= modelo_treinador,
    db=db,                           # ← peça nova: banco persistente
    tools=[TavilyTools(), WikipediaTools()],
    instructions=[
        "Você é o Treinador do ScoutAI FC.",
        "Responda sobre a Seleção Brasileira em português do Brasil, com tom profissional.",
        "Use Tavily para informações recentes, Wikipedia para fatos históricos.",
    ],
    add_history_to_context=True,     # mantém contexto da conversa
    num_history_runs=3,
    markdown=True,
)

### Rodando uma execução e vendo o que ficou no banco



In [ ]:
# Pergunta simples — só pra gerar uma execução guardada
treinador.print_response(
    "Quem é o atual técnico da Seleção Brasileira masculina?",
    user_id="fabio-instrutor",       # identifica o usuário
    session_id="aula-5-1-demo",      # identifica a sessão
    stream=True,
)

### Inspecionando o banco




In [ ]:
import sqlite3

conn = sqlite3.connect("/tmp/scoutai.db")
cursor = conn.cursor()

# Listar tabelas que o Agno criou
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
tabelas = cursor.fetchall()
print("Tabelas no banco:")
for t in tabelas:
    print(f"  - {t[0]}")

conn.close()

Você vai ver tabelas tipo `agno_sessions`, `agno_metrics`, possivelmente outras. **Cada execução do agente alimenta essas tabelas automaticamente** — sem você ter escrito uma linha de SQL.

Ex: tabela de sessões.


In [ ]:
conn = sqlite3.connect("/tmp/scoutai.db")
cursor = conn.cursor()

# Olhar uma das tabelas (exemplo: sessions)
try:
    cursor.execute("SELECT * FROM agno_sessions LIMIT 1")
    rows = cursor.fetchall()
    columns = [desc[0] for desc in cursor.description]
    print("Colunas:", columns[:8], "..." if len(columns) > 8 else "")
    print()
    print("Primeira linha (resumida):")
    for col, val in zip(columns[:8], rows[0][:8] if rows else []):
        valor_resumido = str(val)[:80] + "..." if len(str(val)) > 80 else str(val)
        print(f"  {col}: {valor_resumido}")
except Exception as e:
    print(f"Tabela não encontrada (pode mudar entre versões): {e}")

conn.close()

---

## Passo 4 — Criando o AgentOS

Agora a peça central da aula. `AgentOS` é uma classe que recebe seus agentes (e times, e workflows), e produz um **aplicativo FastAPI completo** pronto pra servir.


In [ ]:
from agno.os import AgentOS

# Cria o AgentOS plugando o agente que já criamos
agent_os = AgentOS(
    agents=[treinador],
)

# Pega o app FastAPI gerado
app = agent_os.get_app()

print(f"AgentOS criado.")
print(f"Tipo do app: {type(app).__name__}")

---

## Passo 5 — A UI do AgentOS

Exemplo local no VS Code

---

### Estado atual do produto

```
ScoutAI FC (estado atual)
│
├── ✅ Arquitetura completa (Aulas 1-4)
│   ├── Time conversacional
│   ├── Workflow estruturado
│   └── Roteador entre os dois
│
├── ✅ Persistência básica (esta aula)
│   ├── SqliteDb conectado
│   ├── AgentOS instanciado
│   └── App FastAPI gerado
│
├── ✅ UI do AgentOS conectada ao servidor (esta aula)
│
├── ❌ Monitoramento programático (via Python)   → Aula 5.2
├── ❌ Memória de longo prazo                   → Aula 5.2
├── ❌ Time e workflow plugados ao AgentOS      → Aula 5.2
├── ❌ Governança, versionamento, escala        → Aula 5.3
└── ❌ Boas práticas para dados pessoais (LGPD) → Aula 5.3
```

### Próxima aula

**Aula 5.2 — Monitorando e gerenciando agentes**

